### Sentiment Analysis - LSTM (Long Short Term Memory)

#### Import Needed Package

In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

#### Load Dataset

In [12]:
data = pd.read_csv('IMDB_Dataset.csv') # read locally

data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [13]:
data.shape

(50000, 2)

In [14]:
type(data)

pandas.core.frame.DataFrame

In [15]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [16]:
data["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

#### One Hot Encoding

In [17]:
# Label Encoder
data.replace({'sentiment': {'positive': 1, 'negative': 0}}, inplace=True)

In [18]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


### Data Preprocessing

In [19]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [20]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [21]:
train_data.shape

(40000, 2)

In [22]:
test_data.shape

(10000, 2)

In [23]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])

In [24]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [25]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [26]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [27]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

### Model Building

In [28]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [31]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 66ms/step - accuracy: 0.7037 - loss: 0.5483 - val_accuracy: 0.8486 - val_loss: 0.3586
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 34s 69ms/step - accuracy: 0.8642 - loss: 0.3335 - val_accuracy: 0.8622 - val_loss: 0.3291
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 71ms/step - accuracy: 0.8837 - loss: 0.2918 - val_accuracy: 0.8650 - val_loss: 0.3248
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 36s 71ms/step - accuracy: 0.8967 - loss: 0.2616 - val_accuracy: 0.8773 - val_loss: 0.2999
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 37s 74ms/step - accuracy: 0.9084 - loss: 0.2262 - val_accuracy: 0.8710 - val_loss: 0.3322


In [32]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8726 - loss: 0.3248


In [33]:
print(loss)

0.31638023257255554


In [34]:
print(accuracy)

0.8761000037193298


### Building predictive system

In [35]:
def predictive_system(review):
    sequences = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequences, maxlen=200)
    prediction = model.predict(padded_sequence)
    sentiment  = "Positive" if prediction[0][0] >= 0.5 else "Negative"
    return sentiment


In [36]:
predictive_system("This movie is great!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step


'Positive'

In [37]:
predictive_system("UIB is a great campus and I love it")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


'Positive'

In [38]:
# Save Model
model.save("tugas.h5")

import joblib
joblib.dump(tokenizer, 'tokenizer.pkl') # dump tokenizer

['tokenizer.pkl']